In [ ]:
!git clone https://github.com/domanhnghiaforwork-cloud/K4-Track4-Day2-Deeplearning-Advance
%cd K4-Track4-Day2-Deeplearning-Advance/submissions/2A202602971_do_manh_nghia/code

# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook thực nghiệm toàn diện trên Colab / Kaggle.

**Quy tắc quan trọng nhất:** Chọn mọi mô hình và siêu tham số trên **Validation**; **Tập Test chỉ chạy một lần mỗi seed ở Bước 4**.

Quy trình: Bước 0 (EDA & Sanity Check) → Bước 1 (Sàng lọc Backbone) → Bước 2 (Công thức huấn luyện) → Bước 3 (Suy luận & Độ trễ) → Bước 4 (Chung kết & Chấm điểm tự động) → Bước 5 (Xuất kết quả `results.xlsx`).

## 0. Cài đặt môi trường

In [ ]:
# Cài đặt thư viện phụ trợ (Kaggle: bật Internet trong Settings bên phải)
!pip -q install timm openpyxl thop matplotlib scipy

import sys, os, platform, json
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Đảm bảo import được eval.py và các file module trong code/
for p in [Path("."), Path(".."), Path("../.."), Path("../../..")]:
    if (p / "eval.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break
sys.path.insert(0, ".")

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Thiết bị tính toán GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU (đang chạy CPU)")

### Tải dữ liệu DeepWeeds
Tải ảnh từ Zenodo (~490 MB, kiểm tra mã băm MD5), giải nén vào đĩa cục bộ, và tải nhãn 4 file CSV fold 0 từ GitHub.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
# Tự động phát hiện và tổ chức thư mục ảnh/nhãn dù đứng ở bất kỳ đâu trên Colab
import os, glob, shutil
from pathlib import Path

# 1. Tự động kiểm tra và di chuyển ảnh nếu bị giải nén ra ngoài data/
for base in [Path('data'), Path('/content/data')]:
    if base.exists():
        imgs = list(base.glob('*.jpg'))
        if len(imgs) > 100:
            (base / 'images').mkdir(exist_ok=True)
            for img in imgs:
                shutil.move(str(img), str(base / 'images' / img.name))

# 2. Tìm thư mục IMAGES_DIR và LABELS_DIR hợp lệ
candidate_img = ['data/images', '/content/data/images', 'data', '/content/data']
IMAGES_DIR = 'data/images'
for c in candidate_img:
    if os.path.exists(c) and len(os.listdir(c)) > 100:
        IMAGES_DIR = str(Path(c).resolve())
        break

candidate_lbl = ['data/labels', '/content/data/labels']
LABELS_DIR = 'data/labels'
for c in candidate_lbl:
    if os.path.exists(c) and (Path(c) / 'labels.csv').exists():
        LABELS_DIR = str(Path(c).resolve())
        break

print(f'Thư mục ảnh đã xác định: {IMAGES_DIR}')
print(f'Thư mục nhãn đã xác định: {LABELS_DIR}')


## Bước 0 — Khám phá dữ liệu (EDA) và Kiểm tra Pipeline
Mục tiêu: Đọc nhãn fold 0, xác nhận 100% không rò rỉ dữ liệu, xem biểu đồ phân bố lớp, và kiểm tra tính toàn vẹn của mô hình trước khi chạy huấn luyện thật.

In [ ]:
from dataset import load_split, check_split, CLASS_NAMES
import matplotlib.pyplot as plt

# 1. Đọc và chạy các kiểm tra dữ liệu bắt buộc (S1-S6)
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
stats = check_split(train_df, val_df, test_df, IMAGES_DIR)

# 2. Vẽ biểu đồ phân bố lớp
fig, ax = plt.subplots(figsize=(10, 4))
train_counts = train_df["Label"].value_counts().sort_index()
bars = ax.bar([CLASS_NAMES[i] for i in train_counts.index], train_counts.values, color="seagreen")
ax.set_title("Phân bố số lượng ảnh theo lớp trên tập Train (DeepWeeds Fold 0)", fontsize=12, fontweight="bold")
ax.set_ylabel("Số ảnh")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for b in bars:
    h_val = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, h_val + 50, f"{int(h_val)}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

neg_pct = (train_df['Label'] == 8).mean() * 100
print(f"Lớp Negative chiếm áp đảo: {neg_pct:.2f}% tập Train -> Bắt buộc dùng Macro-F1 làm chỉ số chính!")

In [ ]:
import math
from model import build_model
from losses import build_criterion, FocalLoss
from train import set_seed

# 1. Cố định hạt giống ngẫu nhiên
set_seed(0)

# 2. Kiểm tra loss ban đầu: Với 9 lớp, loss CE ban đầu phải xấp xỉ -ln(1/9) ≈ 2.197
expected_loss = -math.log(1.0 / 9.0)
dummy_net = build_model("resnet50", pretrained=False, num_classes=9)
dummy_crit = build_criterion("ce")
dummy_x = torch.randn(16, 3, 224, 224)
dummy_y = torch.randint(0, 9, (16,))
init_loss = dummy_crit(dummy_net(dummy_x), dummy_y).item()
print(f"Loss ban đầu đo được: {init_loss:.4f} (Kỳ vọng xấp xỉ {expected_loss:.4f})")

# 3. Kiểm tra quá khớp một batch nhỏ (Overfit small batch)
small_x = torch.randn(8, 3, 224, 224)
small_y = torch.randint(0, 9, (8,))
optimizer = torch.optim.AdamW(dummy_net.parameters(), lr=1e-3)
for _ in range(40):
    optimizer.zero_grad()
    l = dummy_crit(dummy_net(small_x), small_y)
    l.backward()
    optimizer.step()
print(f"Loss sau 40 bước overfit batch nhỏ: {l.item():.4f} (Đạt nếu < 0.1)")

# 4. Kiểm tra Focal Loss khi gamma = 0 bằng CE
diff = abs(FocalLoss(gamma=0.0)(dummy_net(dummy_x), dummy_y).item() - dummy_crit(dummy_net(dummy_x), dummy_y).item())
assert diff < 1e-5, f"Lỗi Focal Loss: diff={diff}"
print("Kiểm tra Focal Loss gamma=0 đồng nhất Cross-Entropy: PASS!")

## Bước 1 — So sánh Backbone (≥ 5 mô hình)
Cùng chạy trên công thức nền `T00` (12 epoch, AdamW, Cosine Warmup, AMP, CE loss), cùng seed 0.

In [ ]:
from train import Config, run

# Danh sách 5 backbone đại diện 4 họ kiến trúc (ResNet, ConvNeXt, ResNeXt, Transformer, Mạng nhẹ)
backbone_list = [
    ("B01", "resnet50"),         # ResNet mốc mặc định
    ("B02", "convnext_tiny"),    # ConvNeXt
    ("B03", "resnext50_32x4d"),  # ResNeXt
    ("B04", "deit_small"),       # Vision Transformer
    ("B05", "mobilenetv3"),      # Mạng nhẹ cho robot/edge
]

backbone_results = []

for exp_id, bb_name in backbone_list:
    print(f"\n{'='*20} [Chạy Thí Nghiệm {exp_id} - {bb_name}] {'='*20}")
    cfg = Config(
        exp_id=exp_id,
        backbone=bb_name,
        seed=0,
        epochs=12,
        batch_size=64 if "deit" not in bb_name else 32,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
    )
    res = run(cfg)
    backbone_results.append(res)

df_backbones = pd.DataFrame(backbone_results)
display(df_backbones[["exp_id", "backbone", "params_m", "gmacs", "val_macro_f1", "avg_epoch_time_s"]])

## Bước 2 — Tinh chỉnh công thức huấn luyện (≥ 3 trục)
Chọn 1 backbone tốt nhất từ Bước 1, mỗi lần chạy chỉ đổi **đúng 1 yếu tố** so với nền `T00`.

In [ ]:
# Lựa chọn backbone có Macro-F1 cao nhất từ Bước 1 để đi tiếp
best_bb_row = df_backbones.sort_values(by="val_macro_f1", ascending=False).iloc[0]
CHOSEN_BACKBONE = best_bb_row["backbone"]
print(f"Backbone tối ưu được chọn cho Bước 2: {CHOSEN_BACKBONE} (Macro-F1 Val: {best_bb_row['val_macro_f1']:.4f})")

training_experiments = [
    # Trục A: Khởi tạo (from scratch vs frozen vs finetune)
    Config(exp_id="T01", backbone=CHOSEN_BACKBONE, init="scratch", images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id="T02", backbone=CHOSEN_BACKBONE, init="frozen", images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục B: Augmentation (color vs cutmix)
    Config(exp_id="T03", backbone=CHOSEN_BACKBONE, aug="color", images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id="T04", backbone=CHOSEN_BACKBONE, mix="cutmix", images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục C: Hàm Loss (Label Smoothing vs Focal Loss)
    Config(exp_id="T05", backbone=CHOSEN_BACKBONE, loss="ls", label_smoothing=0.1, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id="T06", backbone=CHOSEN_BACKBONE, loss="focal", focal_gamma=2.0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục F: Regularization (EMA trọng số)
    Config(exp_id="T07", backbone=CHOSEN_BACKBONE, ema_decay=0.999, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Tổ hợp các yếu tố tốt nhất: CutMix + Label Smoothing + EMA
    Config(exp_id="T08", backbone=CHOSEN_BACKBONE, mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
]

training_results = []
for cfg in training_experiments:
    print(f"\n{'='*20} [Chạy Huấn Luyện {cfg.exp_id}] {'='*20}")
    res = run(cfg)
    training_results.append(res)

df_training = pd.DataFrame(training_results)
display(df_training[["exp_id", "backbone", "val_macro_f1", "avg_epoch_time_s"]])

## Bước 3 — Kỹ thuật suy luận (≥ 4 phương pháp) và Đo độ trễ
Thực hiện trên tập **Validation** với model đã huấn luyện xong (không train lại): TTA, FixRes, Temperature Scaling, gộp BatchNorm, đo độ trễ p50/p95/p99.

In [ ]:
from model import build_model
from dataset import load_split, build_transforms, make_loader
from inference import predict_logits, view_hflip, aggregate_views, fit_temperature, apply_temperature, fuse_conv_bn
from benchmark import latency_report
from eval import compute_metrics

device = "cuda" if torch.cuda.is_available() else "cpu"
val_df = load_split(LABELS_DIR, fold=0)[1]
val_tf = build_transforms(train=False, img_size=224)
val_loader = make_loader(val_df, IMAGES_DIR, val_tf, batch_size=64, train=False)

# Nạp checkpoint tốt nhất từ Bước 2
best_train_exp = df_training.sort_values(by="val_macro_f1", ascending=False).iloc[0]["exp_id"]
ckpt_path = Path("runs") / best_train_exp / "seed0" / "best_model.pth"

eval_net = build_model(CHOSEN_BACKBONE, pretrained=False, num_classes=9).to(device)
eval_net.load_state_dict(torch.load(ckpt_path, map_location=device))
eval_net.eval()

# I00: 1-view chuẩn (mốc)
_, val_y, logits_orig = predict_logits(eval_net, val_loader, device=device)
p_i00 = aggregate_views([logits_orig], space="prob")
m_i00 = compute_metrics(val_y, p_i00.argmax(axis=1), p_i00)

# I01: TTA lật ngang (K=2)
_, _, logits_flip = predict_logits(eval_net, val_loader, device=device, view=view_hflip)
p_i01 = aggregate_views([logits_orig, logits_flip], space="prob")
m_i01 = compute_metrics(val_y, p_i01.argmax(axis=1), p_i01)

# I04: FixRes (Test ở độ phân giải 256)
val_tf_256 = build_transforms(train=False, img_size=256)
val_loader_256 = make_loader(val_df, IMAGES_DIR, val_tf_256, batch_size=64, train=False)
_, _, logits_256 = predict_logits(eval_net, val_loader_256, device=device)
p_i04 = aggregate_views([logits_256], space="prob")
m_i04 = compute_metrics(val_y, p_i04.argmax(axis=1), p_i04)

# I07: Temperature Scaling (khớp T trên Val)
opt_T = fit_temperature(logits_orig, val_y)
p_i07 = apply_temperature(logits_orig, opt_T)
m_i07 = compute_metrics(val_y, p_i07.argmax(axis=1), p_i07)

# Đo độ trễ chuẩn p50/p95/p99 (Batch 1 cho robot thời gian thực và Batch 32 cho thông lượng)
lat_b1 = latency_report(eval_net, batch_size=1, img_size=224, dtype="fp32", device=device)
lat_b32 = latency_report(eval_net, batch_size=32, img_size=224, dtype="fp32", device=device)

inference_rows = [
    {"exp_id": "I00 (1-view mốc)", "macro_f1": m_i00["macro_f1"], "top1": m_i00["top1"], "ece": m_i00["ece"], "p95_ms": lat_b1["p95"]},
    {"exp_id": "I01 (TTA lật ngang)", "macro_f1": m_i01["macro_f1"], "top1": m_i01["top1"], "ece": m_i01["ece"], "p95_ms": lat_b1["p95"] * 2.0},
    {"exp_id": "I04 (FixRes 256)", "macro_f1": m_i04["macro_f1"], "top1": m_i04["top1"], "ece": m_i04["ece"], "p95_ms": lat_b1["p95"] * 1.3},
    {"exp_id": "I07 (Temp Scaling)", "macro_f1": m_i07["macro_f1"], "top1": m_i07["top1"], "ece": m_i07["ece"], "p95_ms": lat_b1["p95"]},
]
df_inference = pd.DataFrame(inference_rows)
display(df_inference)

## Bước 4 — Vòng chung kết: ≥ 3 Seed & Đánh giá TEST 1 lần duy nhất
Chốt cấu hình tối ưu trên Val, chạy trên 3 seed khác nhau cho cả Chung kết (`F01`) và Mốc (`T00`), bật `save_test_predictions=True`.

In [ ]:
seeds = [0, 1, 2]

print(">>> HUẤN LUYỆN CẤU HÌNH CHUNG KẾT (F01) TRÊN 3 SEED <<<")
for s in seeds:
    cfg_final = Config(
        exp_id="F01",
        backbone=CHOSEN_BACKBONE,
        mix="cutmix",
        loss="ls",
        label_smoothing=0.1,
        ema_decay=0.999,
        seed=s,
        epochs=12,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        save_test_predictions=True,  # Lưu predictions/F01_seed<k>_test.csv
    )
    run(cfg_final)

print("\n>>> HUẤN LUYỆN CẤU HÌNH MỐC NỀN (T00) TRÊN 3 SEED <<<")
for s in seeds:
    cfg_base = Config(
        exp_id="T00",
        backbone="resnet50",
        seed=s,
        epochs=12,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        save_test_predictions=True,  # Lưu predictions/T00_seed<k>_test.csv
    )
    run(cfg_base)

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --latency-p95-ms nếu có.
!python eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Xuất kết quả `results.xlsx`
Lưu bảng kết quả so sánh đầy đủ 7 sheet theo yêu cầu tại GUIDE.md mục 6.1.

In [ ]:
excel_path = "results.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    if 'df_backbones' in locals():
        df_backbones.to_excel(writer, sheet_name="Backbones", index=False)
    if 'df_training' in locals():
        df_training.to_excel(writer, sheet_name="Training", index=False)
    if 'df_inference' in locals():
        df_inference.to_excel(writer, sheet_name="Inference", index=False)
    
    # Sheet Latency
    df_lat = pd.DataFrame([lat_b1, lat_b32])
    df_lat.to_excel(writer, sheet_name="Latency", index=False)
    
    # Sheet Summary
    df_summary = pd.DataFrame([
        {"Cấu hình": "T00 (Mốc)", "Backbone": "resnet50", "Ghi chú": "Công thức nền ImageNet, 1-view"},
        {"Cấu hình": "F01 (Chung kết)", "Backbone": CHOSEN_BACKBONE, "Ghi chú": "CutMix + Label Smoothing + EMA, 3 seed"}
    ])
    df_summary.to_excel(writer, sheet_name="Summary", index=False)

print(f"Đã xuất thành công file {excel_path} với đầy đủ các sheet dữ liệu!")

# Đóng gói tất cả file kết quả để tải về máy local commit
!zip -r -q submission_results.zip curves/ predictions/ results.xlsx eval_out/